In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.features.feature_engineer.min_features import _detect_star_players
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.utils.helpers import *
from src.utils.dataScraper import *
from live import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'LAL': ['Jarred Vanderbilt']}

Out Players:
{'DET': ['Kevin Huerter'], 'LAL': ['Luka Doncic']}
Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 4 teams with confirmed lineups
Updated 1 teams with questionable players


### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
p25 = pd.read_csv('data/raw/playoff_stats/P25.csv').sort_values(by='GAME_DATE')
s25 = pd.concat([s25, p25])
s25 = _detect_star_players(s25)

s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
s26 = pd.concat([s26, p26])
s26 = _detect_star_players(s26)

base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,TOP_PLAYER,SECOND_TOP_PLAYER,THIRD_TOP_PLAYER,IS_TOP_STAR,IS_TOP_1_STAR,ACTIVE_STARS_COUNT,TOP_STAR_ACTIVE,TOP_PLAYER_ACTIVE,SECOND_PLAYER_ACTIVE,THIRD_PLAYER_ACTIVE,name
27942,NaN,NaN,NaN,2025-26,1630540,Miles McBride,Miles,1610612752,NYK,New York Knicks,42500213,2026-05-08,NYK @ PHI,W,21.028333,1,6,0.167,1,5,0.2,0,0,0.000,0,0,0,2,0,0,2,0,1,0,3,-4,12.0,0,0,10.0,1,21:02,1,116.6,114.3,114.3,128.0,126.8,126.8,-11.3,-12.5,-12.5,0.125,0.0,25.0,0.000,0.000,0.000,0.0,0.0,0.250,0.250,0.125,0.127,93.36,94.73,78.94,94.73,0.000,42,1.0,6.0,G,4.24,1.60,1.0,0.0,1.0,24.0,0.0,0.0,17.0,0.0,1.0,0.00,1.0,5.0,0.2,1.0,2.0,0.50,38,76,0.500,9,27,0.333,23,32,0.719,13,36,49,25,15.0,6,4,3,21,25,108,14.0,117.3,114.9,101.0,101.1,16.3,13.8,0.658,1.67,18.7,0.349,0.776,0.576,0.160,0.559,0.599,92.6,93.5,77.92,94,0.600,1610612755,PHI,Philadelphia 76ers,36,84,0.429,9,32,0.281,13,16,0.813,9,24,33,23,11.0,7,3,4,25,21,94,-14.0,101.0,101.1,117.3,114.9,-16.3,-13.8,0.639,2.09,18.1,0.224,0.651,0.424,0.118,0.482,0.516,92.6,93.5,77.92,93,0.400,1,SG,25.0,NaN,NaN,3.5,214.5,1,0.142665,0.095110,0.000000,0,4,Karl-Anthony Towns,Jalen Brunson,OG Anunoby,0,0,2,1,1,1,0,Miles McBride
27943,NaN,NaN,NaN,2025-26,1629656,Quentin Grimes,Quentin,1610612755,PHI,Philadelphia 76ers,42500213,2026-05-08,PHI vs. NYK,L,22.466667,2,6,0.333,2,5,0.4,0,0,0.000,1,1,2,2,1,0,0,0,4,1,6,-17,10.4,0,0,12.0,1,22:28,1,77.7,79.1,79.1,121.7,115.9,115.9,-44.0,-36.8,-36.8,0.182,2.0,22.2,0.036,0.050,0.042,11.1,11.1,0.500,0.500,0.140,0.144,91.53,92.94,77.45,92.94,0.008,43,2.0,6.0,NaN,3.97,1.63,5.0,4.0,9.0,19.0,0.0,0.0,12.0,0.0,1.0,0.00,2.0,5.0,0.4,1.0,1.0,1.00,36,84,0.429,9,32,0.281,13,16,0.813,9,24,33,23,11.0,7,3,4,25,21,94,-14.0,101.0,101.1,117.3,114.9,-16.3,-13.8,0.639,2.09,18.1,0.224,0.651,0.424,0.118,0.482,0.516,92.6,93.5,77.92,93,0.400,1610612752,NYK,New York Knicks,38,76,0.500,9,27,0.333,23,32,0.719,13,36,49,25,15.0,6,4,3,21,25,108,14.0,117.3,114.9,101.0,101.1,16.3,13.8,0.658,1.67,18.7,0.349,0.776,0.576,0.160,0.559,0.599,92.6,93.5,77.92,94,0.600,1,SG,25.0,NaN,NaN,-3.5,214.5,1,0.267062,0.089021,0.089021,1,4,Joel Embiid,Tyrese Maxey,Paul George,0,0,3,1,1,1,1,Quen

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_odds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_odds = pd.json_normalize(data)

print("Loaded:", file.name)
team_odds.head()

Loaded: NBA_20260509_143842.json


,home_team,away_team,commence_time,bookmakers
0,Cleveland Cavaliers,Detroit Pistons,2026-05-09 19:14:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Los Angeles Lakers,Oklahoma City Thunder,2026-05-10 00:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,Philadelphia 76ers,New York Knicks,2026-05-10 19:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Minnesota Timberwolves,San Antonio Spurs,2026-05-10 23:40:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = s26
ast_df = s26
reb_df = s26
min_df = s26


#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
# lines_dfs = lines_dfs[lines_dfs['COMMENCE_TIME'] == current_date]
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
# lines_us = lines_us[lines_us['COMMENCE_TIME'] == current_date]
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-05-09 14:34:15
US latest pull: 2026-05-09 14:38:43


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Tobias Harris,Over,21.5,-137,2026-05-09,2026-05-09T21:33:26Z,2026-05-09 14:34:15
1,PrizePicks,player_points,Tobias Harris,Under,21.5,-137,2026-05-09,2026-05-09T21:33:26Z,2026-05-09 14:34:15
2,PrizePicks,player_points,Duncan Robinson,Over,11.5,-137,2026-05-09,2026-05-09T21:33:26Z,2026-05-09 14:34:15
3,PrizePicks,player_points,Duncan Robinson,Under,11.5,-137,2026-05-09,2026-05-09T21:33:26Z,2026-05-09 14:34:15
4,PrizePicks,player_points,Shai Gilgeous-Alexander,Over,29.0,-137,2026-05-10,2026-05-09T21:34:00Z,2026-05-09 14:34:15


### Load my models

In [9]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-05-07.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-05-07.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-05-07.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-05-08.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [11]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
reb_preds.head(10)

[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)
[SKIP] Kelly Oubre Jr: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,RATE_HISTORY,STAT_Q10,STAT_Q50,STAT_Q90
0,Jarrett Allen,REB,23.98,30.16,34.53,0.1545,0.2687,0.4255,"[0.1461038961038961, 0.547945205479452, 0.2144...",3.71,8.10,14.69
1,Deandre Ayton,REB,23.59,27.29,35.89,0.1652,0.2788,0.4190,"[0.320665083135392, 0.0508044030482641, 0.2058...",3.90,7.61,15.04
2,Chet Holmgren,REB,26.02,32.58,36.23,0.1527,0.2569,0.3846,"[0.3722084367245657, 0.3015075376884422, 0.194...",3.97,8.37,13.93
3,LeBron James,REB,33.08,37.92,41.09,0.0958,0.1703,0.2807,"[0.3027245206861756, 0.1632208922742111, 0.230...",3.17,6.46,11.53
4,Shai Gilgeous-Alexander,REB,29.39,32.60,39.08,0.0487,0.1204,0.2324,"[0.1721170395869191, 0.0541125541125541, 0.067...",1.43,3.92,9.08
5,Austin Reaves,REB,33.35,38.83,40.87,0.0574,0.1181,0.2338,"[0.1537785588752196, 0.0747011952191235, 0.077...",1.91,4.59,9.55
6,Ajay Mitchell,REB,23.92,31.74,36.89,0.0441,0.1137,0.2432,"[0.0658544616397761, 0.1179554390563564, 0.118...",1.06,3.61,8.97
7,Jaxson Hayes,REB,8.55,14.77,21.69,0.0916,0.2167,0.3904,"[0.144115292233787, 0.2757986669731096, 0.3069...",0.78,3.20,8.47
8,Marcus Smart,REB,27.91,34.57,38.38,0.0391,0.0975,0.2174,"[0.0849858356940509, 0.0, 0.0887573964497041, ...",1.09,3.37,8.34
9,Cason Wallace,REB,19.42,27.10,31.85,0.0443,0.1130,0.2389,"[0.0449438202247191, 0.1647107267860819, 0.163...",0.86,3.06,7.61


In [12]:
from live import adjust_predictions

# Build contexts dict once (using the notebook's get_game_context)
game_contexts = {
    name: get_game_context(base_df, name, team_odds, is_playoff=True)
    for name in pts_preds["PLAYER_NAME"]
}

# Adjust the model's Q50 predictions with scenario signals
pts_preds = adjust_predictions(pts_preds, base_df, game_contexts)
ast_preds = adjust_predictions(ast_preds, base_df, game_contexts)
reb_preds = adjust_predictions(reb_preds, base_df, game_contexts)
pts_preds.head()

Tobias Harris [PTS] [ix: away_underdog]  pace_bucket=middle_pace  MIN: 36.5→38.7 (Δ+2.17)  RATE: 0.4873→0.4733 (Δ-0.0140)
Duncan Robinson [PTS] [ix: away_underdog]  pace_bucket=middle_pace  MIN: 35.5→34.5 (Δ-1.04)  RATE: 0.3922→0.3850 (Δ-0.0072)
Shai Gilgeous-Alexander [PTS] [ix: away_low_pace]  pace_bucket=low_pace  MIN: 32.6→32.7 (Δ+0.10)  RATE: 0.7613→0.6468 (Δ-0.1144)
LeBron James [PTS] [ix: home_low_pace]  pace_bucket=low_pace  MIN: 37.9→39.8 (Δ+1.84)  RATE: 0.6188→0.5666 (Δ-0.0522)
Austin Reaves [PTS] [ix: home_low_pace]  pace_bucket=low_pace  MIN: 38.8→40.1 (Δ+1.23)  RATE: 0.6066→0.4880 (Δ-0.1186)
Chet Holmgren [PTS] [ix: away_low_pace]  pace_bucket=low_pace  MIN: 32.6→32.7 (Δ+0.10)  RATE: 0.5049→0.5048 (Δ-0.0002)
Ajay Mitchell [PTS]  pace_bucket=low_pace  MIN: 31.7→33.6 (Δ+1.86)  RATE: 0.4908→0.4878 (Δ-0.0030)
Rui Hachimura [PTS] [ix: home_low_pace]  pace_bucket=low_pace  MIN: 37.1→38.5 (Δ+1.38)  RATE: 0.3681→0.3736 (Δ+0.0055)
Marcus Smart [PTS]  pace_bucket=low_pace  MIN: 34.6

,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,RATE_HISTORY,STAT_Q10,STAT_Q50,STAT_Q90,ADJ_CONTEXT_OK,ADJ_CONTEXT_ERR,ADJ_ACTIVE_STARS,ADJ_STARS_MISSING,ADJ_SPREAD_ROLE,ADJ_CTX_SPREAD,ADJ_CTX_TOTAL,ADJ_MIN_DELTA,ADJ_RATE_DELTA,ADJ_MIN_SHIFT,ADJ_RATE_SHIFT,ADJ_USED_INTERACTION,ADJ_MIN_LOG,ADJ_RATE_LOG
0,Tobias Harris,PTS,31.45,38.69,41.86,0.2339,0.4733,0.7228,"[0.350372, 0.312975, 0.701631, 0.417997, 0.307...",7.36,18.31,30.26,True,None,3,0,underdog,1.5,224.5,2.1708,-0.01400,2.17,-0.0140,True,"{'stars': (1.7811, 78), 'pace': (-0.1978, 53),...","{'stars': (-0.0186, 78), 'pace': (-0.0148, 53)..."
1,Duncan Robinson,PTS,25.72,34.47,36.92,0.1626,0.3850,0.6221,"[0.64717, 0.515307, 0.462702, 0.812982, 0.7320...",4.18,13.27,22.97,True,None,3,0,underdog,1.5,224.5,-1.0423,-0.00725,-1.04,-0.0072,True,"{'stars': (-0.8439, 56), 'pace': (-1.4013, 56)...","{'stars': (-0.006, 56), 'pace': (0.0077, 56), ..."
2,Shai Gilgeous-Alexander,PTS,29.49,32.70,39.18,0.4087,0.6468,0.8659,"[0.642865, 0.778407, 0.733487, 0.700398, 1.045...",12.05,21.15,33.93,True,None,3,0,favorite,-9.5,210.5,0.0995,-0.11445,0.10,-0.1145,True,"{'stars': (0.3413, 77), 'pace': (0.0, 55), 'in...","{'stars': (-0.0287, 77), 'pace': (-0.0528, 55)..."
3,LeBron James,PTS,34.92,39.76,42.93,0.3303,0.5666,0.8277,"[0.583521, 0.404818, 0.448121, 0.72266, 0.7654...",11.53,22.53,35.53,True,None,3,0,underdog,9.5,210.5,1.8358,-0.05220,1.84,-0.0522,True,"{'stars': (-0.184, 54), 'pace': (0.6048, 47), ...","{'stars': (-0.0703, 54), 'pace': (-0.007, 47),..."
4,Austin Reaves,PTS,34.59,40.06,42.10,0.2634,0.4880,0.7175,"[0.584388, 0.254906, 0.2448, 0.331588, 0.53021...",9.11,19.55,30.21,True,None,3,0,underdog,9.5,210.5,1.2350,-0.11860,1.23,-0.1186,True,"{'stars': (0.6612, 53), 'pace': (-0.0197, 42),...","{'stars': (-0.0436, 53), 'pace': (-0.091, 42),..."


### Get Line Probabilities

In [9]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.head()

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
0,Shai Gilgeous-Alexander,AST,4.5,31.48,37.38,40.08,19.72,23.02,23.74,0.951,0.049
1,Austin Reaves,AST,3.5,32.59,37.79,41.92,17.38,19.78,19.16,0.711,0.289
2,Luguentz Dort,AST,1.5,21.03,27.97,33.23,10.66,13.52,14.66,0.687,0.313
3,Jalen Brunson,AST,6.5,30.83,34.60,40.53,17.65,19.39,21.15,0.629,0.371
4,Tyrese Maxey,AST,6.0,26.14,32.91,43.25,14.86,17.89,21.84,0.502,0.498


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
underdog_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
underdog_all_lines.head()

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
7,Joel Embiid,AST,4.5,26.26,31.36,40.54,15.40,17.41,20.49,0.757,0.243,AST,Underdog,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-110.0,-105.0,0.524,0.512,4.9,6.0,2.85,0.4,1.5,-0.140,0.556,0.444,6.15,-13.31,0.8,0.6,0.53,0.40,34.24,4.76,0.34,0.04,4.00,4.0
28,Quentin Grimes,REB,2.5,15.05,21.94,27.71,7.52,10.15,10.91,0.732,0.268,REB,Underdog,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-110.0,-105.0,0.524,0.512,3.1,3.0,1.37,0.6,0.5,-0.438,0.669,0.331,27.72,-35.38,0.4,0.7,0.67,0.69,22.52,4.15,0.14,0.05,3.29,7.0
30,Rudy Gobert,REB,11.5,27.60,31.78,38.88,18.02,20.47,23.75,0.589,0.411,REB,Underdog,San Antonio Spurs,4.5,216.5,110.4,3.0,100.72,12.0,102.0,-115.0,0.495,0.535,11.2,11.0,3.01,-0.3,-0.5,0.100,0.460,0.540,-7.08,0.96,0.6,0.5,0.53,0.46,33.07,4.64,0.11,0.04,9.57,7.0
38,De'Aaron Fox,REB,3.5,27.44,32.93,37.80,12.94,14.55,13.92,0.569,0.431,REB,Underdog,Minnesota Timberwolves,-4.5,216.5,112.5,8.0,101.50,10.0,-115.0,100.0,0.535,0.500,3.5,3.5,2.07,0.0,0.0,0.000,0.500,0.500,-6.52,0.00,0.4,0.5,0.53,0.57,33.18,3.68,0.25,0.03,3.88,8.0
62,Joel Embiid,PTS,26.5,26.26,31.36,40.54,13.36,23.81,40.82,0.464,0.536,PTS,Underdog,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-103.0,-110.0,0.507,0.524,26.9,27.5,7.46,0.4,1.0,-0.054,0.522,0.478,2.88,-8.75,0.4,0.5,0.53,0.55,34.24,4.76,0.34,0.04,22.75,4.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
prizePicks_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
prizePicks_all_lines.head(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
0,Shai Gilgeous-Alexander,AST,4.5,31.48,37.38,40.08,19.72,23.02,23.74,0.951,0.049,AST,PrizePicks,Los Angeles Lakers,-8.5,215.2,115.5,20.0,99.22,22.0,100.0,108.0,0.500,0.481,7.4,7.5,2.07,2.9,3.0,-1.401,0.919,0.081,83.80,-83.15,1.0,0.9,0.87,0.80,33.04,5.57,0.32,0.04,7.71,7.0
1,Austin Reaves,AST,3.5,32.59,37.79,41.92,17.38,19.78,19.16,0.711,0.289,AST,PrizePicks,Oklahoma City Thunder,8.5,215.2,106.5,1.0,100.37,16.0,-115.0,110.0,0.535,0.476,5.1,5.0,2.42,1.6,1.5,-0.661,0.746,0.254,39.47,-46.66,0.6,0.8,0.87,0.71,34.86,5.12,0.26,0.04,4.00,7.0
2,Luguentz Dort,AST,1.5,21.03,27.97,33.23,10.66,13.52,14.66,0.687,0.313,AST,PrizePicks,Los Angeles Lakers,-8.5,215.2,115.5,20.0,99.22,22.0,-115.0,-115.0,0.535,0.535,1.5,1.5,1.35,0.0,0.0,0.000,0.500,0.500,-6.52,-6.52,0.6,0.5,0.40,0.38,23.03,2.75,0.12,0.03,1.86,7.0
3,Jalen Brunson,AST,6.5,30.83,34.60,40.53,17.65,19.39,21.15,0.629,0.371,AST,PrizePicks,Philadelphia 76ers,1.5,214.5,114.4,17.0,100.39,15.0,-137.0,-137.0,0.578,0.578,6.5,7.0,3.50,0.5,1.0,-0.143,0.557,0.443,-3.64,-23.36,0.4,0.6,0.67,0.56,34.74,3.70,0.32,0.05,5.25,8.0
4,Tyrese Maxey,AST,6.0,26.14,32.91,43.25,14.86,17.89,21.84,0.502,0.498,AST,PrizePicks,New York Knicks,-1.5,214.5,112.3,7.0,97.71,25.0,-137.0,-137.0,0.578,0.578,5.4,5.5,2.46,-0.6,-0.5,0.244,0.404,0.596,-30.11,3.10,0.2,0.3,0.40,0.46,37.86,6.29,0.27,0.05,4.86,7.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
betr_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
93,James Harden,PTS,18.5,31.47,37.88,42.27,9.94,20.40,36.28,0.537,0.463,PTS,Betr DFS,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-122.0,-103.0,0.550,0.507,21.3,20.5,4.08,2.8,2.0,-0.686,0.754,0.246,37.20,-51.52,0.4,0.7,0.53,0.71,35.19,4.93,0.27,0.06,22.40,10.0
111,Ajay Mitchell,PTS,15.5,23.68,33.18,38.39,4.29,14.08,26.24,0.366,0.634,PTS,Betr DFS,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-115.0,-106.0,0.535,0.515,11.5,9.5,4.74,-4.0,-6.0,0.844,0.199,0.801,-62.80,55.67,0.2,0.1,0.13,0.22,25.78,6.70,0.19,0.06,9.25,4.0
39,Evan Mobley,REB,8.5,25.64,33.17,39.57,3.52,7.64,13.62,0.503,0.497,REB,Betr DFS,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-113.0,-103.0,0.531,0.507,9.1,7.5,4.23,0.6,-1.0,-0.142,0.556,0.444,4.80,-12.49,0.6,0.4,0.47,0.56,31.69,5.46,0.22,0.05,9.14,14.0
28,Julius Randle,REB,6.5,26.56,35.36,40.93,1.65,5.43,11.24,0.430,0.570,REB,Betr DFS,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,110.0,-130.0,0.476,0.565,6.7,7.0,2.31,0.2,0.5,-0.087,0.535,0.465,12.35,-17.73,0.6,0.6,0.53,0.54,33.35,2.97,0.27,0.04,6.29,7.0
119,Cason Wallace,PTS,6.5,9.53,15.75,22.80,0.73,5.51,14.92,0.305,0.695,PTS,Betr DFS,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-128.0,105.0,0.561,0.488,6.8,6.0,4.34,0.3,-0.5,-0.069,0.528,0.472,-5.95,-3.24,0.2,0.4,0.40,0.52,21.87,3.16,0.14,0.05,7.43,7.0


In [14]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_odds,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

_m = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left')
draftKings_all_lines = _m.dropna(subset=[c for c in _m.columns if not str(c).startswith('ADJ_')])
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
25,Quentin Grimes,REB,2.5,12.27,19.21,27.07,0.49,2.12,5.94,0.565,0.435,REB,DraftKings Pick6,New York Knicks,7.5,212.5,112.3,7.0,97.71,25.0,130.0,-115.0,0.435,0.535,3.2,3.0,1.23,0.7,0.5,-0.569,0.715,0.285,64.45,-46.72,0.6,0.7,0.73,0.69,22.74,4.31,0.17,0.07,3.67,6.0
26,Rudy Gobert,REB,11.5,21.35,30.40,36.47,2.47,6.47,13.26,0.126,0.874,REB,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,110.0,-116.0,0.476,0.537,10.9,11.0,3.28,-0.6,-0.5,0.183,0.427,0.573,-10.33,6.70,0.6,0.5,0.60,0.46,32.82,4.90,0.11,0.04,9.50,6.0
72,OG Anunoby,PTS,14.5,27.95,35.76,41.20,5.32,15.77,29.23,0.638,0.362,PTS,DraftKings Pick6,Philadelphia 76ers,-7.5,212.5,114.4,17.0,100.40,15.0,100.0,-130.0,0.500,0.565,19.7,20.0,8.90,5.2,5.5,-0.584,0.720,0.280,44.00,-50.46,0.8,0.7,0.67,0.64,33.30,7.34,0.18,0.05,18.29,7.0
3,Julius Randle,AST,4.5,26.56,35.36,40.93,0.66,3.65,6.91,0.297,0.703,AST,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,129.0,-135.0,0.437,0.574,4.2,4.0,1.55,-0.3,-0.5,0.194,0.423,0.577,-3.13,0.44,0.6,0.4,0.33,0.47,33.35,2.97,0.27,0.04,5.14,7.0
77,Anthony Edwards,PTS,20.5,16.76,23.86,32.55,2.64,9.12,23.30,0.103,0.897,PTS,DraftKings Pick6,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,115.0,-122.0,0.465,0.550,21.8,20.5,11.56,0.3,-1.0,-0.026,0.510,0.490,9.65,-10.84,0.6,0.5,0.53,0.72,30.48,7.93,0.31,0.04,28.00,7.0


In [15]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
51,Isaiah Hartenstein,REB,8.5,9.32,16.88,23.18,1.59,5.15,10.54,0.259,0.741,REB,PrizePicks,Los Angeles Lakers,-15.8,213.5,115.5,20.0,99.22,22.0,-137.0,-137.0,0.578,0.578,8.8,8.0,3.97,0.8,0.0,-0.202,0.580,0.420,0.34,-27.34,0.4,0.4,0.47,0.56,21.14,4.87,0.14,0.03,9.57,7.0
41,Ausar Thompson,REB,7.0,24.75,32.87,39.53,3.14,7.07,12.91,0.660,0.340,REB,Betr DFS,Orlando Magic,-8.5,202.0,113.6,13.0,100.56,14.0,-137.0,-137.0,0.578,0.578,7.5,7.5,3.37,0.5,0.5,-0.148,0.559,0.441,-3.30,-23.71,0.8,0.5,0.40,0.23,30.03,5.87,0.14,0.04,7.92,13.0
44,Donovan Mitchell,REB,4.0,30.13,36.45,41.75,2.12,4.86,9.86,0.803,0.197,REB,PrizePicks,Toronto Raptors,-8.5,210.5,112.1,5.0,99.22,21.0,-137.0,-137.0,0.578,0.578,5.3,5.5,1.42,1.3,1.5,-0.915,0.820,0.180,41.85,-68.86,0.8,0.8,0.60,0.46,34.73,3.00,0.30,0.06,4.42,12.0
114,Marcus Smart,PTS,10.5,30.59,39.25,45.10,5.65,17.72,33.50,0.766,0.234,PTS,Underdog,Oklahoma City Thunder,15.8,213.5,106.5,1.0,100.37,16.0,105.0,-115.0,0.488,0.535,11.4,10.0,7.28,0.9,-0.5,-0.124,0.549,0.451,12.54,-15.68,0.6,0.5,0.47,0.39,31.33,6.19,0.18,0.06,14.00,2.0
29,Naz Reid,REB,6.5,17.35,24.38,30.98,1.52,4.71,10.71,0.379,0.621,REB,Betr DFS,San Antonio Spurs,14.0,216.5,110.4,3.0,100.72,12.0,130.0,-140.0,0.435,0.583,6.3,7.0,2.36,-0.2,0.5,0.085,0.466,0.534,7.18,-8.46,0.8,0.6,0.53,0.39,24.88,4.94,0.21,0.03,5.57,7.0


### Get top EVs for 2 legs

In [16]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 87  |  Pairs: 885  |  Slate: 10  |  STRONG: 1  |  MARGINAL: 9  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 26  |  Pairs: 105  |  Slate: 6  |  STRONG: 0  |  MARGINAL: 6  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 33  |  Pairs: 90  |  Slate: 6  |  STRONG: 0  |  MARGINAL: 6  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings.json


In [19]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 77  |  Pairs: 655  |  Slate: 10  |  STRONG: 0  |  MARGINAL: 10  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr.json


### Top EVs for 3 Legs

In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 87  |  Triples: 18373  |  Slate: 10  |  STRONG: 0  |  MARGINAL: 9  |  SKIP: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\prizepicks_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 26  |  Triples: 623  |  Slate: 5  |  STRONG: 0  |  MARGINAL: 5  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\underdog_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 77  |  Triples: 12278  |  Slate: 9  |  STRONG: 0  |  MARGINAL: 9  |  SKIP: 0  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\betr_3leg.json


In [23]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 33  |  Triples: 502  |  Slate: 4  |  STRONG: 0  |  MARGINAL: 3  |  SKIP: 1  |  JSON: C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\data\props\ev_analysis\draftKings_3leg.json
